# ML Pipelines & Column Transformers: Production-Ready Workflows

Building robust machine learning workflows requires more than just good models — it demands **reproducible, leak-proof, and maintainable pipelines**. This notebook covers everything from basic chaining to advanced grid search over pipeline parameters.

In [ ]:
# core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# preprocessing & pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# model selection
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV

# metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# persistence
import joblib

# utils
from sklearn.base import BaseEstimator, TransformerMixin
import scipy.stats as stats

print('All imports successful.')

---
## 1. The Problem: Why Pipelines?

Without pipelines, ML workflows suffer from three critical issues:

### Code Duplication
Every time you transform features for training, testing, or new data, you repeat the same steps — error-prone and hard to maintain.

### Data Leakage in Cross-Validation
If you scale or impute *before* splitting, the test fold's statistics leak into the training set, giving overly optimistic results.

### Reproducibility
Scattered transformations are nearly impossible to reproduce in production.

In [ ]:
# BAD PRACTICE - manual transform before split causes data leakage
rng = np.random.RandomState(42)
X_demo = pd.DataFrame({'feature': rng.randn(100) * 10 + 50})
y_demo = (X_demo['feature'] > 50).astype(int)

# Leakage: scaling uses global mean/std (includes unseen data)
X_scaled = (X_demo - X_demo.mean()) / X_demo.std()
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_scaled, y_demo, test_size=0.2, random_state=42)
print('Scaled before split - test data influenced the scaler!')
print(f'Train mean: {X_train_l.values.mean():.3f}, Test mean: {X_test_l.values.mean():.3f} (should differ more)')

---
## 2. Pipeline Basics

In [ ]:
# Build a simple pipeline: impute -> scale -> logistic regression
simple_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42))
])

simple_pipe

In [ ]:
# Fit and predict - one call, no leakage
X_demo2 = pd.DataFrame({'feature': [1, 2, np.nan, 4, 5, 6, np.nan, 8, 9, 10]})
y_demo2 = pd.Series([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

simple_pipe.fit(X_demo2, y_demo2)
preds = simple_pipe.predict(X_demo2)
print(f'Accuracy: {accuracy_score(y_demo2, preds):.2f}')

In [ ]:
# Accessing pipeline steps
print('named_steps keys:', list(simple_pipe.named_steps.keys()))
print('Via __getitem__:', simple_pipe['scaler'])
print('Mean from imputer:', simple_pipe.named_steps['imputer'].statistics_)

---
## 3. ColumnTransformer: Different Transforms for Different Columns

In [ ]:
# ColumnTransformer applies distinct preprocessing to specified column groups
demo_df = pd.DataFrame({
    'age': [25, 30, np.nan, 40, 35],
    'income': [50000, 60000, 55000, np.nan, 48000],
    'education': ['BS', 'MS', 'BS', 'PhD', 'MS'],
    'city': ['NYC', 'LA', 'NYC', 'LA', 'SF']
})

ct_demo = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['age', 'income']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False))
    ]), ['education', 'city'])
])

ct_demo.fit_transform(demo_df)

In [ ]:
# Using make_column_selector for automatic column type detection
ct_selector = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), make_column_selector(dtype_include=np.number)),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', sparse_output=False))
    ]), make_column_selector(dtype_include=object))
])

ct_selector.fit_transform(demo_df)

---
## 4. FeatureUnion: Combining Multiple Feature Extraction Pipelines

In [ ]:
# FeatureUnion runs parallel pipelines and concatenates results
fu_demo = FeatureUnion([
    ('dense_features', 'passthrough'),
    ('sqrt_transform', FunctionTransformer(func=np.sqrt, validate=True))
])

X_fu = np.array([[4.0, 9.0], [16.0, 25.0]])
fu_demo.fit_transform(X_fu)

---
## 5. Hyperparameter Tuning Over Pipelines

In [ ]:
# Use __ double underscore to nest into pipeline step parameters
pipe_tune = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

param_grid = {
    'scaler__with_mean': [True, False],
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear']
}

X_grid = np.random.randn(100, 5)
y_grid = (X_grid[:, 0] + X_grid[:, 1] > 0).astype(int)

gs = GridSearchCV(pipe_tune, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_grid, y_grid)

print('Best params:', gs.best_params_)
print(f'Best CV score: {gs.best_score_:.4f}')

In [ ]:
# RandomizedSearchCV for larger parameter spaces
rng_dist = {
    'clf__C': stats.loguniform(0.001, 100),
    'clf__penalty': ['l1', 'l2'],
}

rs = RandomizedSearchCV(pipe_tune, rng_dist, n_iter=20, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
rs.fit(X_grid, y_grid)

print('Best params:', rs.best_params_)
print(f'Best CV score: {rs.best_score_:.4f}')

---
## 6. Caching Transformers with `memory`

In [ ]:
# The memory parameter caches fitted transformers to disk - speeds up grid search re-runs
from tempfile import mkdtemp
cachedir = mkdtemp()

pipe_cached = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
], memory=cachedir)

gs_cached = GridSearchCV(pipe_cached, {'clf__C': [0.1, 1, 10]}, cv=3)
gs_cached.fit(X_grid, y_grid)
print(f'Best score: {gs_cached.best_score_:.4f} - cached for future calls.')

---
## 7. Real-World Example: Titanic Survival Prediction

In [ ]:
# Load Titanic data
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)
titanic.head()

In [ ]:
print('Shape:', titanic.shape)
print('Missing values:', titanic.isnull().sum().to_dict())
print('Dtypes:', titanic.dtypes.to_dict())

In [ ]:
# Feature engineering: create some derived columns
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['IsAlone'] = (titanic['FamilySize'] == 1).astype(int)
titanic['Title'] = titanic['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
titanic['Title'] = titanic['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
titanic['Title'] = titanic['Title'].replace('Mlle', 'Miss')
titanic['Title'] = titanic['Title'].replace('Ms', 'Miss')
titanic['Title'] = titanic['Title'].replace('Mme', 'Mrs')

# Select features for modeling
feature_cols = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone', 'Title']
X = titanic[feature_cols]
y = titanic['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Define numeric and categorical pipelines
numeric_features = ['Age', 'Fare', 'FamilySize']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine in ColumnTransformer
titanic_ct = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Full pipeline with classifier
titanic_pipe = Pipeline([
    ('preprocessor', titanic_ct),
    ('clf', RandomForestClassifier(random_state=42))
])

titanic_pipe

In [ ]:
# Fit and evaluate
titanic_pipe.fit(X_train, y_train)
y_pred = titanic_pipe.predict(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('Classification Report:', classification_report(y_test, y_pred, output_dict=True))

In [ ]:
# Grid search over preprocessing + model hyperparameters
titanic_param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'preprocessor__cat__imputer__strategy': ['most_frequent', 'constant'],
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [3, 5, None],
    'clf__min_samples_split': [2, 5]
}

titanic_gs = GridSearchCV(titanic_pipe, titanic_param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
titanic_gs.fit(X_train, y_train)

print('Best params:', titanic_gs.best_params_)
print(f'Best CV accuracy: {titanic_gs.best_score_:.4f}')
print(f'Test accuracy: {accuracy_score(y_test, titanic_gs.predict(X_test)):.4f}')

---
## 8. Saving and Loading Pipelines

In [ ]:
# Save the full pipeline (preprocessing + model) as a single file
joblib.dump(titanic_gs.best_estimator_, 'titanic_pipeline.pkl')
print('Pipeline saved to titanic_pipeline.pkl')

In [ ]:
# Load and predict on new data with no re-fit needed
loaded_pipe = joblib.load('titanic_pipeline.pkl')

# Simulate a single new passenger
new_passenger = pd.DataFrame([{
    'Pclass': 3, 'Sex': 'male', 'Age': 25, 'Fare': 8.05,
    'Embarked': 'S', 'FamilySize': 2, 'IsAlone': 0, 'Title': 'Mr'
}])

prediction = loaded_pipe.predict(new_passenger)[0]
probability = loaded_pipe.predict_proba(new_passenger)[0][1]
print(f'Predicted survival: {prediction} (probability: {probability:.3f})')

---
## 9. Custom Transformers

In [ ]:
# FunctionTransformer wraps any function into a transformer
def clip_outliers(X, lower=0.01, upper=0.99):
    q_low = np.percentile(X, lower * 100, axis=0)
    q_high = np.percentile(X, upper * 100, axis=0)
    return np.clip(X, q_low, q_high)

clip_transformer = FunctionTransformer(clip_outliers, kw_args={'lower': 0.05, 'upper': 0.95})

X_clip_demo = np.array([[1, 1000], [2, 2000], [3, 3000], [100, 4000]])
print('Before:', X_clip_demo)
print('After:', clip_transformer.fit_transform(X_clip_demo))

In [ ]:
# Custom transformer class (more flexible: learn parameters during fit)
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, date_column='date'):
        self.date_column = date_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        dates = pd.to_datetime(X[self.date_column])
        X['year'] = dates.dt.year
        X['month'] = dates.dt.month
        X['day'] = dates.dt.day
        X['dayofweek'] = dates.dt.dayofweek
        X['quarter'] = dates.dt.quarter
        return X.drop(columns=[self.date_column])

# Demo
dates_df = pd.DataFrame({'date': ['2023-01-15', '2023-06-20', '2024-12-25'], 'value': [10, 20, 30]})
DateFeatureExtractor('date').fit_transform(dates_df)

---
## 10. Real-World Example 2: Customer Churn with Mixed Data Types

In [ ]:
# Generate synthetic customer churn dataset
rng = np.random.RandomState(42)
n = 1000

churn_df = pd.DataFrame({
    'age': rng.randint(18, 70, n).astype(float),
    'tenure_months': rng.randint(1, 72, n).astype(float),
    'monthly_charges': rng.uniform(20, 120, n),
    'total_charges': rng.uniform(100, 8000, n),
    'gender': rng.choice(['Male', 'Female'], n),
    'internet_service': rng.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.3, 0.4, 0.3]),
    'contract': rng.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'payment_method': rng.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
    'feedback_text': [f"{'good' if rng.random() < 0.6 else 'bad'} service {'love' if rng.random() < 0.5 else 'hate'}" for _ in range(n)],
    'churn': rng.binomial(1, 0.27, n)
})

# Introduce missing values
churn_df.loc[rng.choice(n, 50), 'monthly_charges'] = np.nan
churn_df.loc[rng.choice(n, 30), 'internet_service'] = np.nan

churn_df.head()

In [ ]:
# Define three column groups: numeric, categorical, and text
num_cols = ['age', 'tenure_months', 'monthly_charges', 'total_charges']
cat_cols = ['gender', 'internet_service', 'contract', 'payment_method']
text_col = 'feedback_text'

# Build pipelines for each type
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False))
])

text_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50))
])

# ColumnTransformer with mixed types
churn_ct = ColumnTransformer([
    ('numeric', num_pipe, num_cols),
    ('categorical', cat_pipe, cat_cols),
    ('text', text_pipe, text_col)
])

# Final pipeline
churn_pipe = Pipeline([
    ('preprocessor', churn_ct),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

churn_pipe

In [ ]:
# Train-validation split and evaluation
X_c = churn_df.drop('churn', axis=1)
y_c = churn_df['churn']

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)
churn_pipe.fit(Xc_train, yc_train)

yc_pred = churn_pipe.predict(Xc_test)
print(f'Churn model test accuracy: {accuracy_score(yc_test, yc_pred):.4f}')
print(classification_report(yc_test, yc_pred, target_names=['No churn', 'Churn']))

In [ ]:
# RandomizedSearchCV on the churn pipeline
churn_dist = {
    'preprocessor__numeric__impute__strategy': ['mean', 'median'],
    'preprocessor__text__tfidf__max_features': [20, 50, 100],
    'clf__C': stats.loguniform(0.01, 10),
    'clf__penalty': ['l1', 'l2']
}

churn_rs = RandomizedSearchCV(churn_pipe, churn_dist, n_iter=30, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1)
churn_rs.fit(Xc_train, yc_train)

print('Best params:', churn_rs.best_params_)
print(f'Best CV ROC-AUC: {churn_rs.best_score_:.4f}')
print(f'Test ROC-AUC: {roc_auc_score(yc_test, churn_rs.predict_proba(Xc_test)[:, 1]):.4f}')

In [ ]:
# Save the tuned churn pipeline
joblib.dump(churn_rs.best_estimator_, 'churn_pipeline.pkl')
print('Churn pipeline saved.')

---
## Practice Exercises

**Exercise 1 - Build a Pipeline**

Create a pipeline that imputes missing values with the median, applies `StandardScaler`, and trains a `RandomForestClassifier` on the Iris dataset (use numeric columns only). Evaluate with 5-fold cross-validation.

**Exercise 2 - ColumnTransformer with Mixed Types**

Load the Titanic dataset from seaborn (`sns.load_dataset('titanic')`). Use `ColumnTransformer` to:
- Impute `age` and `fare` with median, then scale
- Impute `embarked` and `sex` with most frequent, then one-hot encode
- Train a `LogisticRegression` and tune `C` via `GridSearchCV`

**Exercise 3 - Custom Transformer for Outlier Capping**

Write a custom transformer class `OutlierCapper` that inherits `BaseEstimator` and `TransformerMixin`. It should learn the 1st and 99th percentiles during `fit` and clip values outside that range during `transform`. Integrate it into a pipeline with `StandardScaler` and `LogisticRegression`, then search over the percentile thresholds via `GridSearchCV`.

**Exercise 4 - FeatureUnion + GridSearch**

Use `make_column_selector` to split the California housing dataset into numeric columns. Then create a `FeatureUnion` that computes:
1. The original features passed through
2. A `FunctionTransformer` that squares each feature
3. Another `FunctionTransformer` that takes the log of absolute values

Fit a `Ridge` regression model and tune `alpha` over `[0.1, 1, 10]`.